# **Start Section:**


In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d09_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_ewqi train
    shutil.copy(glob.glob("/kaggle/input/**/d09_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_ewqi test
    print("Kaggle: water_quality_ewqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_ewqi data copied to ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


In [4]:
# Go to find & replace button and replace (water_quality_ewqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.


In [5]:
feature_names = ['pH', 'TDS', 'EC', 'TH', 'ca', 'mg', 'CO3', 'HCO3', 'NA', 'K', 'CHLORIDE', 'SULPHATE', 'NITRATE', 'FLUORIDE']
TARGET_LABEL = 'Pre-monsoon EWQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)


In [6]:
train_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")


Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))



Shape of training data: (47, 15)
First 5 rows of training data:
      pH   TDS    EC    TH   ca    mg  CO3  HCO3   NA    K  CHLORIDE  SULPHATE  \
0  7.90   552   865   452  272   180  0.2   4.5  260   44       375       681   
1  7.52  1985  2256  1054  529   525  0.0  15.0  272   47       728       324   
2  7.65  1865  2032   636  371   265  0.0   9.0  373   25       713       503   
3  7.25  4855  7856  2050  795  1255  0.0  12.5  335  106        81       676   
4  7.96  2855  3696   865  310   555  0.3   4.6  184  106       698       324   

   NITRATE  FLUORIDE  Pre monsoon EWQI   
0       60      0.40              70.52  
1       10      0.65             138.78  
2       54      0.85             113.07  
3       17      0.69             266.32  
4       25      1.20             158.05  

Shape of test data: (47, 15)
First 5 rows of test data:
      pH   TDS    EC    TH     ca      mg  CO3  HCO3   NA    K  CHLORIDE  \
0  8.38   625   975   320  120.0   200.0  0.0   5.8  280   35 

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)



Shape of X_train: (47, 14)
Shape of y_train: (47,)
Shape of X_test: (47, 14)
Shape of y_test: (47,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])



First five rows of normalized X_train:
[[ 0.3797749  -1.15230942 -0.94309903 -0.47923131 -0.09229732 -0.56852482
  -0.03922784 -1.18768739 -0.02713192 -0.87884263 -0.2770724   1.15873613
   1.41276153 -1.25316406]
 [-0.78765707 -0.2953837  -0.52059702  0.45803078  1.31357822  0.10564733
  -0.50015494  1.61405127  0.07159325 -0.77136925  1.29937074 -0.45273785
  -1.61366471 -0.53733141]
 [-0.38827245 -0.36714301 -0.58863473 -0.19275917  0.44926563 -0.40242444
  -0.50015494  0.01305775  0.90253007 -1.55950739  1.23238307  0.35525611
   1.04959038  0.03533472]
 [-1.61714822  1.42085972  1.18034563  2.00871688  2.76868675  1.53215652
  -0.50015494  0.94697063  0.58990037  1.34227396 -1.59003071  1.13616647
  -1.18996504 -0.42279818]
 [ 0.56410627  0.22487128 -0.08321177  0.16377408  0.11557533  0.16427099
   0.19123571 -1.16100417 -0.6523913   1.34227396  1.1653954  -0.45273785
  -0.70573684  1.03750043]]

First five rows of normalized X_test:
[[ 1.85442582 -1.10865585 -0.90968766 -0.6847

# **Functions:**


In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']


In [12]:
# FIX M4: one miscoverage level for the conformal notebooks. AlphaNet LEARNS its own level alpha~,
# so ALPHA is only the nominal reference value reported next to it.
ALPHA = 0.1


# FIX A1: lambda is the size/coverage trade-off chosen by the user (Gauthier, Bach & Jordan 2026,
# eq. 9). The official code reports every lambda separately and never compares losses across
# lambdas. This table evaluates, for EVERY lambda, its best run (lowest final training loss within
# that lambda) on the test set, so the trade-off can be seen.
def _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test, y_test):
    rows = []
    for lam, res in sorted(all_results.items(), key=lambda kv: kv[0]):
        final_losses = res["all_losses"][:, -1]
        r = int(np.argmin(final_losses))
        ev = evaluate_acp(model=model, alpha_net=res["models"][r], X_calib=X_calib, y_calib=y_calib,
                          X_test=X_test, y_test=y_test)
        a = ev.get("alpha_hat", ev.get("alpha"))
        rows.append({
            "lambda": lam,
            "best_run (1-based)": r + 1,
            "final_train_loss": float(final_losses[r]),
            "alpha_learned": a,
            "Target_coverage": 1.0 - a,
            "test_coverage": ev["coverage"],
            "width": ev["interval_width"],
            "Miss_over_alpha": (1.0 - ev["coverage"]) / a,
        })
    return pd.DataFrame(rows)


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _to_numpy_2d(x):
    arr = x.to_numpy() if hasattr(x, "to_numpy") else np.asarray(x)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr.astype(np.float32)

def _to_numpy_1d(y):
    arr = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    return arr.reshape(-1).astype(np.float32)

def _ensure_excel_workbook(excel_file_path):
    if not excel_file_path:
        return
    os.makedirs(os.path.dirname(excel_file_path), exist_ok=True)
    if not os.path.exists(excel_file_path):
        wb = Workbook()
        wb.active.title = "Init"
        wb.save(_ensure_parent_dir(excel_file_path))

def save_plot_to_excel(fig, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    with io.BytesIO() as buf:
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        img = Image(PImage.open(buf))
        workbook = load_workbook(_ensure_excel_file(excel_file_path))
        base_name = sheet_name
        i = 1
        while sheet_name in workbook.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        worksheet = workbook.create_sheet(title=sheet_name)
        worksheet.add_image(img, "A1")
        workbook.save(_ensure_parent_dir(excel_file_path))

def save_values_to_excel(results_dict, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    df = pd.DataFrame(results_dict)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

def save_dataframe_to_excel(df, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

class AlphaNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        # FIX A2: start the output near alpha~ = 1 (sigmoid(5) ~ 0.993), as in the official code
        # (github.com/GauthierE/adaptive-coverage-policies, initialize_to_one), so alpha~ starts in the
        # region alpha~ > 1/(n+1) where the width formula is valid. Was: random initial output.
        with torch.no_grad():
            self.net[2].weight.data.normal_(0, 0.01)
            self.net[2].bias.data.fill_(5.0)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def interval_size(scores, alpha, eps=1e-6):
    n = scores.numel()
    denominator = torch.clamp(alpha * (n + 1) - 1.0, min=eps)
    return 2.0 * scores.sum() / denominator

def _compute_scores_without_index(base_model, X_calib, y_calib, holdout_idx):
    loo_X = np.delete(X_calib, holdout_idx, axis=0)
    loo_y = np.delete(y_calib, holdout_idx, axis=0)
    return np.abs(base_model.predict(loo_X) - loo_y)

def _build_loo_feature_matrix(base_model, X_calib, y_calib):
    rows = []
    for idx in range(len(X_calib)):
        scores = _compute_scores_without_index(base_model, X_calib, y_calib, idx)
        rows.append([scores.sum()])
    return torch.tensor(np.asarray(rows, dtype=np.float32))

def _resolve_model_params(model_class, best_params):
    params = dict(best_params or {})
    if "TabNet" in model_class.__name__ and "verbose" in params:
        params.pop("verbose", None)
    try:
        model_class(**params)
        return params
    except Exception as e:
        # FIX B5: was a silent fallback to default parameters
        print(f"[Warning] {model_class.__name__}: tuned parameters rejected ({type(e).__name__}: {e}); "
              "falling back to the model's DEFAULT parameters")
        return {}

def _train_alpha_net(
    base_model,
    X_calib,
    y_calib,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    learning_rate=1e-2,  # FIX A3 (was 1e-3)
    min_steps=2000,  # FIX A3
    alpha_clip=1e-3,
    seed=42,
):
    X_train_alpha = _build_loo_feature_matrix(base_model, X_calib, y_calib)
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []
        for run_idx in range(num_runs):
            local_seed = seed + run_idx
            torch.manual_seed(local_seed)
            np.random.seed(local_seed)
            random.seed(local_seed)
            dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
            loader = DataLoader(dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)
            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=learning_rate)
            all_losses = []
            all_sizes = []
            all_alphas = []
            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []
                alpha_net.train()
                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)
                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached
                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + lambda_reg * alpha_pred).mean()
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))
                all_losses.append(float(np.mean(epoch_losses)))
                all_sizes.append(float(np.mean(epoch_sizes)))
                all_alphas.append(float(np.mean(epoch_alphas)))
            alpha_net.eval()
            lambda_losses.append(np.asarray(all_losses, dtype=np.float32))
            lambda_sizes.append(np.asarray(all_sizes, dtype=np.float32))
            lambda_alphas.append(np.asarray(all_alphas, dtype=np.float32))
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)
        all_results[lambda_reg] = {
            "all_losses": np.vstack(lambda_losses),
            "all_sizes": np.vstack(lambda_sizes),
            "all_alphas": np.vstack(lambda_alphas),
            "models": lambda_models,
        }
    return all_results

def _moving_average(values, window=10):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    if window <= 1 or arr.size < window:
        return arr.copy()
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(arr, kernel, mode="valid")

def _smooth_runs(run_matrix, window=10):
    smoothed_runs = [_moving_average(run_curve, window=window) for run_curve in np.asarray(run_matrix)]
    if not smoothed_runs:
        return np.array([]), np.array([])
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.array([]), np.array([])
    stacked = np.asarray([curve[:min_len] for curve in smoothed_runs], dtype=float)
    return stacked.mean(axis=0), stacked.std(axis=0)

def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
    for idx, (lambda_reg, result_dict) in enumerate(sorted(all_results.items(), key=lambda item: item[0])):
        color = colors[idx % len(colors)]
        loss_mean, loss_std = _smooth_runs(result_dict["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result_dict["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result_dict["all_alphas"], window=10)
        if loss_mean.size == 0:
            continue
        epochs_axis = np.arange(1, len(loss_mean) + 1)
        axes[0].plot(epochs_axis, loss_mean, color=color, label=f"lambda={lambda_reg}")
        axes[0].fill_between(epochs_axis, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)
        axes[1].plot(epochs_axis, size_mean, color=color, label=f"lambda={lambda_reg}")
        axes[1].fill_between(epochs_axis, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)
        axes[2].plot(epochs_axis, alpha_mean, color=color, label=f"lambda={lambda_reg}")
        axes[2].fill_between(epochs_axis, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)
    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean Size")
    axes[2].set_title(r"Smoothed Mean $\tilde{\alpha}$")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel(r"Mean $\tilde{\alpha}$")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize="small")
    plt.tight_layout()
    return fig

def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    final_losses = all_results[lambda_main]["all_losses"][:, -1]
    run_idx = int(np.argmin(final_losses))
    return lambda_main, run_idx, all_results[lambda_main]["models"][run_idx]


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    calib_scores = np.abs(model.predict(X_calib_np) - y_calib_np)
    test_feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(test_feature), min=1e-3, max=1.0 - 1e-3).item())
    score_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(interval_size(score_tensor, torch.tensor(alpha_hat, device=DEVICE)).item())
    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width
    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))
    return {
        "X_test": X_test_np,
        "y_test": y_test_np,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "alpha": alpha_hat,
        "interval_width": interval_width,
        "coverage": coverage,
        "rmse": rmse,
        "mae": mae,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }

def conformal_predictions_ACP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    alpha=ALPHA,  # FIX M4: was 0.1 (nominal only; AlphaNet learns alpha~)
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.35,
        random_state=42,
    )
    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)
    all_results = _train_alpha_net(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )
    best_lambda, best_run_idx, alpha_net = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    eval_bundle = evaluate_acp(
        model=model,
        alpha_net=alpha_net,
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )
    y_true = eval_bundle["y_test"]
    y_pred = eval_bundle["y_pred"]
    lower = eval_bundle["lower"]
    upper = eval_bundle["upper"]
    print(f"{model_name} ACP best lambda: {best_lambda}")
    print(f"{model_name} ACP best run: {best_run_idx + 1}")
    print(f"{model_name} ACP alpha: {eval_bundle['alpha']:.4f}")
    print(f"{model_name} ACP width: {eval_bundle['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {eval_bundle['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {eval_bundle['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {eval_bundle['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {eval_bundle['target_coverage'] * 100:.2f}%  |  miss / alpha~: {eval_bundle['miss_over_alpha']:.3f}  # FIX A1")
    fig_pred, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))
    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    plt.tight_layout()
    fig_dynamics = plot_acp_training_dynamics(all_results)
    if excel_file_path:
        save_plot_to_excel(fig_pred, excel_file_path, "conformal_predictions_ACP")
        save_plot_to_excel(fig_dynamics, excel_file_path, "acp_training_dynamics")
        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), eval_bundle["alpha"]),
                "ACP_interval_width": np.full(len(y_true), eval_bundle["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_best_lambda": np.full(len(y_true), best_lambda),
                "ACP_best_run": np.full(len(y_true), best_run_idx + 1),
                "ACP_target_coverage": np.full(len(y_true), eval_bundle["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), eval_bundle["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )
        training_summary_rows = []
        training_curve_rows = []
        for lambda_reg, result_dict in sorted(all_results.items(), key=lambda item: item[0]):
            final_losses = result_dict["all_losses"][:, -1]
            final_sizes = result_dict["all_sizes"][:, -1]
            final_alphas = result_dict["all_alphas"][:, -1]
            training_summary_rows.append(
                {
                    "lambda": lambda_reg,
                    "runs": result_dict["all_losses"].shape[0],
                    "best_run": int(np.argmin(final_losses)) + 1,
                    "final_loss_mean": float(np.mean(final_losses)),
                    "final_loss_std": float(np.std(final_losses)),
                    "final_size_mean": float(np.mean(final_sizes)),
                    "final_size_std": float(np.std(final_sizes)),
                    "final_alpha_mean": float(np.mean(final_alphas)),
                    "final_alpha_std": float(np.std(final_alphas)),
                }
            )
            mean_loss_curve = result_dict["all_losses"].mean(axis=0)
            mean_size_curve = result_dict["all_sizes"].mean(axis=0)
            mean_alpha_curve = result_dict["all_alphas"].mean(axis=0)
            for epoch_idx in range(len(mean_loss_curve)):
                training_curve_rows.append(
                    {
                        "lambda": lambda_reg,
                        "epoch": epoch_idx + 1,
                        "mean_loss": float(mean_loss_curve[epoch_idx]),
                        "mean_size": float(mean_size_curve[epoch_idx]),
                        "mean_alpha": float(mean_alpha_curve[epoch_idx]),
                    }
                )
        save_dataframe_to_excel(pd.DataFrame(training_summary_rows), excel_file_path, "acp_training_summary")
        save_dataframe_to_excel(pd.DataFrame(training_curve_rows), excel_file_path, "acp_training_curves")
        save_dataframe_to_excel(_acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np),
                                excel_file_path, "acp_lambda_comparison")  # FIX A1
    plt.close(fig_pred)
    plt.close(fig_dynamics)
    eval_bundle["model"] = model
    eval_bundle["alpha_net"] = alpha_net
    eval_bundle["all_results"] = all_results
    eval_bundle["best_lambda"] = best_lambda
    eval_bundle["best_run"] = best_run_idx + 1
    return eval_bundle


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

def _train_alpha_net_multi(
    base_model,
    X_calib,
    y_calib,
    X_train_alpha,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    lr=1e-2,  # FIX A3 (was 1e-3)
    alpha_clip=1e-3,
    min_steps=2000,  # FIX A3
    seed=42,
):
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    Dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
    loader = DataLoader(Dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)

    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []

        for run_idx in range(num_runs):
            torch.manual_seed(seed + run_idx)
            np.random.seed(seed + run_idx)

            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=lr)

            run_losses = []
            run_sizes = []
            run_alphas = []

            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                alpha_net.train()
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []

                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)

                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached

                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + float(lambda_reg) * alpha_pred).mean()

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))

                run_losses.append(float(np.mean(epoch_losses)))
                run_sizes.append(float(np.mean(epoch_sizes)))
                run_alphas.append(float(np.mean(epoch_alphas)))

            alpha_net.eval()
            lambda_losses.append(run_losses)
            lambda_sizes.append(run_sizes)
            lambda_alphas.append(run_alphas)
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)

        all_results[float(lambda_reg)] = {
            "all_losses": np.asarray(lambda_losses, dtype=np.float32),
            "all_sizes": np.asarray(lambda_sizes, dtype=np.float32),
            "all_alphas": np.asarray(lambda_alphas, dtype=np.float32),
            "models": lambda_models,
        }

    return all_results


def _moving_average(values, window=10):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(values, kernel, mode="valid")


def _smooth_runs(run_matrix, window=10):
    run_matrix = np.asarray(run_matrix, dtype=np.float32)
    if run_matrix.ndim == 1:
        run_matrix = run_matrix.reshape(1, -1)
    if run_matrix.size == 0:
        return np.asarray([]), np.asarray([])

    smoothed_runs = [_moving_average(run, window=window) for run in run_matrix]
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.asarray([]), np.asarray([])

    trimmed = np.vstack([curve[:min_len] for curve in smoothed_runs])
    return trimmed.mean(axis=0), trimmed.std(axis=0)


def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if not all_results:
        for ax in axes:
            ax.text(0.5, 0.5, "No training results", ha="center", va="center")
            ax.axis("off")
        return fig

    sorted_results = sorted(all_results.items(), key=lambda item: item[0])
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sorted_results)))

    for color, (lambda_reg, result) in zip(colors, sorted_results):
        loss_mean, loss_std = _smooth_runs(result["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result["all_alphas"], window=10)

        if loss_mean.size:
            x_loss = np.arange(1, loss_mean.size + 1)
            axes[0].plot(x_loss, loss_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[0].fill_between(x_loss, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)

        if size_mean.size:
            x_size = np.arange(1, size_mean.size + 1)
            axes[1].plot(x_size, size_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[1].fill_between(x_size, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)

        if alpha_mean.size:
            x_alpha = np.arange(1, alpha_mean.size + 1)
            axes[2].plot(x_alpha, alpha_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[2].fill_between(x_alpha, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)

    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Size")

    axes[2].set_title("Smoothed Mean Alpha-tilde")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Alpha-tilde")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize="small")

    plt.tight_layout()
    return fig


def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    losses = all_results[lambda_main]["all_losses"]
    if losses.size == 0:
        raise RuntimeError("AlphaNet training produced no valid runs.")
    final_losses = losses[:, -1]
    run_idx = int(np.argmin(final_losses))
    return {
        "lambda": lambda_main,
        "run_idx": run_idx,
        "final_loss": float(final_losses[run_idx]),
        "model": all_results[lambda_main]["models"][run_idx],
    }


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    calib_preds = np.asarray(model.predict(X_calib_np)).reshape(-1)
    calib_scores = np.abs(calib_preds - y_calib_np)

    feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(feature), min=1e-3, max=1.0 - 1e-3).item())

    scores_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(
        interval_size(scores_tensor, torch.tensor(alpha_hat, dtype=torch.float32, device=DEVICE)).item()
    )

    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width

    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))

    return {
        "coverage": coverage,
        "interval_width": interval_width,
        "alpha_hat": alpha_hat,
        "rmse": rmse,
        "mae": mae,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "y_test": y_test_np,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }


def run_full_acp_pipeline(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np, y_train_np, test_size=0.35, random_state=42
    )

    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)

    X_train_alpha = _build_loo_feature_matrix(model, X_calib, y_calib)
    all_results = _train_alpha_net_multi(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        X_train_alpha=X_train_alpha,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )

    dynamics_fig = plot_acp_training_dynamics(all_results)
    dynamics_fig.suptitle(f"ACP Training Dynamics - {model_name}")
    dynamics_fig.tight_layout(rect=[0, 0, 1, 0.95])
    if excel_file_path:
        save_plot_to_excel(dynamics_fig, excel_file_path, "acp_training_dynamics")
    plt.close(dynamics_fig)

    best_choice = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    metrics = evaluate_acp(
        model=model,
        alpha_net=best_choice["model"],
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )

    y_true = metrics["y_test"]
    y_pred = metrics["y_pred"]
    lower = metrics["lower"]
    upper = metrics["upper"]

    print(f"{model_name} best lambda: {best_choice['lambda']}")
    print(f"{model_name} best run: {best_choice['run_idx']}")
    print(f"{model_name} ACP alpha_hat: {metrics['alpha_hat']:.4f}")
    print(f"{model_name} ACP width: {metrics['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {metrics['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {metrics['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {metrics['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {metrics['target_coverage'] * 100:.2f}%  |  miss / alpha~: {metrics['miss_over_alpha']:.3f}  # FIX A1")

    fig, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))

    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    plt.tight_layout()

    if excel_file_path:
        save_plot_to_excel(fig, excel_file_path, "conformal_predictions_ACP")

        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), metrics["alpha_hat"]),
                "ACP_interval_width": np.full(len(y_true), metrics["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_target_coverage": np.full(len(y_true), metrics["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), metrics["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )

        save_values_to_excel({
                "model_name": [model_name],
                "best_lambda": [best_choice["lambda"]],
                "best_run": [best_choice["run_idx"]],
                "best_final_loss": [best_choice["final_loss"]],
                "coverage": [metrics["coverage"]],
                "interval_width": [metrics["interval_width"]],
                "rmse": [metrics["rmse"]],
                "mae": [metrics["mae"]],
                "alpha_hat": [metrics["alpha_hat"]],
                "Target_coverage": [metrics["target_coverage"]],  # FIX A1
                "Miss_over_alpha": [metrics["miss_over_alpha"]],  # FIX A1
            },
            excel_file_path,
            "acp_final_metrics",
        )

        summary = {
            "lambda": [],
            "mean_final_loss": [],
            "std_final_loss": [],
            "mean_final_size": [],
            "std_final_size": [],
            "mean_final_alpha": [],
            "std_final_alpha": [],
        }
        curve_data = {
            "lambda": [],
            "run": [],
            "epoch": [],
            "loss": [],
            "size": [],
            "alpha_tilde": [],
        }

        for lambda_reg, result in sorted(all_results.items(), key=lambda item: item[0]):
            losses = result["all_losses"]
            sizes = result["all_sizes"]
            alphas = result["all_alphas"]

            summary["lambda"].append(lambda_reg)
            summary["mean_final_loss"].append(float(losses[:, -1].mean()))
            summary["std_final_loss"].append(float(losses[:, -1].std()))
            summary["mean_final_size"].append(float(sizes[:, -1].mean()))
            summary["std_final_size"].append(float(sizes[:, -1].std()))
            summary["mean_final_alpha"].append(float(alphas[:, -1].mean()))
            summary["std_final_alpha"].append(float(alphas[:, -1].std()))

            n_runs, n_epochs = losses.shape
            for run_idx in range(n_runs):
                for epoch_idx in range(n_epochs):
                    curve_data["lambda"].append(lambda_reg)
                    curve_data["run"].append(run_idx)
                    curve_data["epoch"].append(epoch_idx + 1)
                    curve_data["loss"].append(float(losses[run_idx, epoch_idx]))
                    curve_data["size"].append(float(sizes[run_idx, epoch_idx]))
                    curve_data["alpha_tilde"].append(float(alphas[run_idx, epoch_idx]))

        save_values_to_excel(summary, excel_file_path, "acp_training_summary")
        save_values_to_excel(curve_data, excel_file_path, "acp_training_curves")
        lambda_df = _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np)  # FIX A1
        print(lambda_df.to_string(index=False))
        save_dataframe_to_excel(lambda_df, excel_file_path, "acp_lambda_comparison")

    plt.close(fig)

    return {
        "model": model,
        "all_results": all_results,
        "best_choice": best_choice,
        "metrics": metrics,
    }


In [15]:
# Unified ACP execution entry point:
# run_full_acp_pipeline(model_class, best_params, X_train, y_train, X_test, y_test, model_name, excel_file_path)


# **Tuned Parameters**


In [16]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/water-quality-ewqi-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with Lightgbm**


In [17]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')


In [18]:
run_full_acp_pipeline(
    LGBMRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "LightGBM",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/LightGBM.xlsx",
)


LightGBM best lambda: 50
LightGBM best run: 0
LightGBM ACP alpha_hat: 0.9990
LightGBM ACP width: 49.9694
LightGBM ACP coverage: 63.83%
LightGBM ACP RMSE: 30.9726
LightGBM ACP MAE: 24.3452
LightGBM ACP target coverage (1 - alpha~): 0.10%  |  miss / alpha~: 0.362  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   3         59.959564          0.999            0.001       0.638298 49.969387         0.362064
   20.0                   2         69.949562          0.999            0.001       0.638298 49.969387         0.362064
   50.0                   1         99.919571          0.999            0.001       0.638298 49.969387         0.362064


{'model': LGBMRegressor(bagging_freq=0, colsample_bytree=0.5, learning_rate=0.05,
               max_depth=3, min_child_samples=5, min_child_weight=1e-05,
               n_estimators=400, n_jobs=-1, num_leaves=63,
               objective='regression', random_state=42, reg_alpha=0,
               reg_lambda=0.1, subsample=0.7, verbose=-1),
 'all_results': {10.0: {'all_losses': array([[60.185303, 60.141865, 60.10493 , ..., 59.959564, 59.95957 ,
           59.95957 ],
          [60.2001  , 60.169033, 60.141026, ..., 59.95957 , 59.959568,
           59.959568],
          [60.211174, 60.176163, 60.14511 , ..., 59.959568, 59.959568,
           59.959564]], dtype=float32),
   'all_sizes': array([[50.247276, 50.193905, 50.148506, ..., 49.96957 , 49.969574,
           49.969574],
          [50.265438, 50.22728 , 50.19287 , ..., 49.96957 , 49.969574,
           49.96957 ],
          [50.27904 , 50.23605 , 50.19791 , ..., 49.969574, 49.969566,
           49.96957 ]], dtype=float32),
   'all_alph

# **Conformal Predictions with XGBoost**


In [19]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')


In [20]:
run_full_acp_pipeline(
    XGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "XGBoost",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/XGBoost.xlsx",
)


XGBoost best lambda: 50
XGBoost best run: 2
XGBoost ACP alpha_hat: 0.9297
XGBoost ACP width: 41.4306
XGBoost ACP coverage: 59.57%
XGBoost ACP RMSE: 27.0522
XGBoost ACP MAE: 20.9685
XGBoost ACP target coverage (1 - alpha~): 7.03%  |  miss / alpha~: 0.435  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         48.379128       0.999000         0.001000       0.531915 38.388992         0.468554
   20.0                   2         58.369133       0.999000         0.001000       0.531915 38.388992         0.468554
   50.0                   3         87.859718       0.929738         0.070262       0.595745 41.430569         0.434806


{'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.5, colsample_bynode=None, colsample_bytree=0.7,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=0.5, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...),
 'all_results': {10.0: {'all_losses': array([[48.540504, 48.50942 , 48.482994, ..., 48.37913 , 48.379128,
           48.379128],
          [48.55109 , 48.52886 , 48.50882 , ..., 48.37913 , 48.379135,
           48.37913 ],
          [48.55902

# **Conformal Predictions with GPBoost**


In [21]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')


In [22]:
run_full_acp_pipeline(
    GPBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GPBoost",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/GPBoost.xlsx",
)


GPBoost best lambda: 50
GPBoost best run: 0
GPBoost ACP alpha_hat: 0.9990
GPBoost ACP width: 50.1709
GPBoost ACP coverage: 59.57%
GPBoost ACP RMSE: 31.3956
GPBoost ACP MAE: 24.7880
GPBoost ACP target coverage (1 - alpha~): 0.10%  |  miss / alpha~: 0.405  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         60.161091          0.999            0.001       0.595745 50.170906          0.40466
   20.0                   3         70.151085          0.999            0.001       0.595745 50.170906          0.40466
   50.0                   1        100.121094          0.999            0.001       0.595745 50.170906          0.40466


{'model': GPBoostRegressor(colsample_bytree=0.5, learning_rate=0.05, max_depth=3,
                  min_child_samples=5, min_child_weight=1e-05, n_estimators=500,
                  random_state=42, reg_alpha=0.1, reg_lambda=0, subsample=0.6,
                  verbose=-1),
 'all_results': {10.0: {'all_losses': array([[60.387947, 60.344296, 60.307186, ..., 60.1611  , 60.16109 ,
           60.161095],
          [60.402817, 60.371605, 60.343456, ..., 60.161095, 60.161095,
           60.16109 ],
          [60.413948, 60.37877 , 60.34757 , ..., 60.161095, 60.16109 ,
           60.161095]], dtype=float32),
   'all_sizes': array([[50.449913, 50.396343, 50.350754, ..., 50.17109 , 50.171093,
           50.17109 ],
          [50.46816 , 50.42984 , 50.395298, ..., 50.171093, 50.171093,
           50.171093],
          [50.48182 , 50.43866 , 50.40037 , ..., 50.171093, 50.171093,
           50.17109 ]], dtype=float32),
   'all_alphas': array([[0.99380326, 0.9947961 , 0.9956426 , ..., 0.999     , 0.9

# **Conformal Predictions with NGBoost**


In [23]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')


In [24]:
run_full_acp_pipeline(
    NGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "NGBoost",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/NGBoost.xlsx",
)


NGBoost best lambda: 50
NGBoost best run: 2
NGBoost ACP alpha_hat: 0.9816
NGBoost ACP width: 44.3457
NGBoost ACP coverage: 72.34%
NGBoost ACP RMSE: 28.7058
NGBoost ACP MAE: 19.8725
NGBoost ACP target coverage (1 - alpha~): 1.84%  |  miss / alpha~: 0.282  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         53.517616       0.999000         0.001000       0.723404 43.527458         0.276873
   20.0                   1         63.507618       0.999000         0.001000       0.723404 43.527458         0.276873
   50.0                   3         93.364655       0.981591         0.018409       0.723404 44.345741         0.281783


{'model': NGBRegressor(learning_rate=0.1, minibatch_frac=0.7,
              random_state=RandomState(MT19937) at 0x7C018566DA40, verbose=0),
 'all_results': {10.0: {'all_losses': array([[53.707542, 53.670963, 53.63987 , ..., 53.517616, 53.51762 ,
           53.517616],
          [53.72    , 53.693836, 53.670258, ..., 53.517616, 53.51762 ,
           53.51762 ],
          [53.72931 , 53.699795, 53.673626, ..., 53.51762 , 53.51762 ,
           53.51762 ]], dtype=float32),
   'all_sizes': array([[43.76951 , 43.723003, 43.683445, ..., 43.52762 , 43.52762 ,
           43.52762 ],
          [43.78534 , 43.75208 , 43.7221  , ..., 43.52762 , 43.527615,
           43.52762 ],
          [43.797188, 43.759674, 43.726406, ..., 43.52762 , 43.52762 ,
           43.52762 ]], dtype=float32),
   'all_alphas': array([[0.99380314, 0.99479586, 0.9956425 , ..., 0.999     , 0.999     ,
           0.999     ],
          [0.99346596, 0.9941755 , 0.99481606, ..., 0.999     , 0.999     ,
           0.999     ],

# **Conformal Predictions with Gradient Boosting**


In [25]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


In [26]:
run_full_acp_pipeline(
    GradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GradientBoosting",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/Gradient Boosting.xlsx",
)


GradientBoosting best lambda: 50
GradientBoosting best run: 0
GradientBoosting ACP alpha_hat: 0.9669
GradientBoosting ACP width: 43.3758
GradientBoosting ACP coverage: 53.19%
GradientBoosting ACP RMSE: 29.1926
GradientBoosting ACP MAE: 24.1463
GradientBoosting ACP target coverage (1 - alpha~): 3.31%  |  miss / alpha~: 0.484  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         51.891991        0.99900          0.00100       0.531915 41.901836         0.468554
   20.0                   3         61.881989        0.99900          0.00100       0.531915 41.901836         0.468554
   50.0                   1         91.675423        0.96694          0.03306       0.531915 43.375843         0.484089


{'model': GradientBoostingRegressor(ccp_alpha=0.01, criterion='squared_error',
                           learning_rate=0.2, max_depth=7, max_features='log2',
                           max_leaf_nodes=50, min_impurity_decrease=0.1,
                           min_samples_leaf=5, min_weight_fraction_leaf=0.01,
                           random_state=42, tol=0.001),
 'all_results': {10.0: {'all_losses': array([[52.072895, 52.038086, 52.00848 , ..., 51.891994, 51.891994,
           51.89199 ],
          [52.08475 , 52.05985 , 52.037407, ..., 51.891994, 51.892   ,
           51.891994],
          [52.09366 , 52.065586, 52.0407  , ..., 51.891994, 51.891994,
           51.891994]], dtype=float32),
   'all_sizes': array([[42.134865, 42.090122, 42.05205 , ..., 41.901993, 41.901993,
           41.901993],
          [42.150097, 42.118103, 42.089252, ..., 41.901993, 41.901993,
           41.901993],
          [42.16153 , 42.125484, 42.093506, ..., 41.90199 , 41.901993,
           41.901993]], dtyp

# **Conformal Predictions with CatBoost**


In [27]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')


In [28]:
run_full_acp_pipeline(
    CatBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "CatBoost",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/CatBoost.xlsx",
)


CatBoost best lambda: 50
CatBoost best run: 1
CatBoost ACP alpha_hat: 0.9990
CatBoost ACP width: 49.9979
CatBoost ACP coverage: 76.60%
CatBoost ACP RMSE: 28.0726
CatBoost ACP MAE: 18.7228
CatBoost ACP target coverage (1 - alpha~): 0.10%  |  miss / alpha~: 0.234  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         59.988102          0.999            0.001       0.765957 49.997921         0.234277
   20.0                   2         69.978096          0.999            0.001       0.765957 49.997921         0.234277
   50.0                   2         99.948105          0.999            0.001       0.765957 49.997921         0.234277


{'model': <catboost.core.CatBoostRegressor at 0x7c015b960a50>,
 'all_results': {10.0: {'all_losses': array([[60.21401 , 60.170547, 60.133587, ..., 59.988106, 59.988106,
           59.988106],
          [60.228806, 60.197723, 60.169716, ..., 59.988102, 59.988106,
           59.988102],
          [60.23994 , 60.204906, 60.17383 , ..., 59.988102, 59.988102,
           59.988102]], dtype=float32),
   'all_sizes': array([[50.27597 , 50.222584, 50.17716 , ..., 49.99811 , 49.998104,
           49.99811 ],
          [50.294144, 50.255974, 50.221558, ..., 49.998108, 49.99811 ,
           49.998104],
          [50.307827, 50.2648  , 50.226627, ..., 49.998104, 49.998108,
           49.998108]], dtype=float32),
   'all_alphas': array([[0.99380314, 0.994796  , 0.9956426 , ..., 0.999     , 0.999     ,
           0.999     ],
          [0.99346596, 0.9941752 , 0.9948156 , ..., 0.999     , 0.999     ,
           0.999     ],
          [0.99321175, 0.99401015, 0.99472   , ..., 0.999     , 0.999     ,
 

# **Conformal Predictions with HistGradientBoosting**


In [29]:
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')


In [30]:
run_full_acp_pipeline(
    HistGradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "HGBR",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/HGBM.xlsx",
)


HGBR best lambda: 50
HGBR best run: 1
HGBR ACP alpha_hat: 0.9990
HGBR ACP width: 56.7695
HGBR ACP coverage: 78.72%
HGBR ACP RMSE: 24.6361
HGBR ACP MAE: 19.1889
HGBR ACP target coverage (1 - alpha~): 0.10%  |  miss / alpha~: 0.213  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         66.759750          0.999            0.001       0.787234 56.769543         0.212979
   20.0                   2         76.749748          0.999            0.001       0.787234 56.769543         0.212979
   50.0                   2        106.719757          0.999            0.001       0.787234 56.769543         0.212979


{'model': HistGradientBoostingRegressor(early_stopping=False, max_bins=64, max_depth=7,
                               max_leaf_nodes=15, min_samples_leaf=5,
                               n_iter_no_change=5, random_state=42,
                               validation_fraction=0.2),
 'all_results': {10.0: {'all_losses': array([[67.02329 , 66.972595, 66.92949 , ..., 66.75976 , 66.75975 ,
           66.75975 ],
          [67.04056 , 67.004326, 66.97168 , ..., 66.75975 , 66.75976 ,
           66.75975 ],
          [67.05353 , 67.012665, 66.97645 , ..., 66.75976 , 66.75976 ,
           66.75975 ]], dtype=float32),
   'all_sizes': array([[57.085255, 57.02464 , 56.973064, ..., 56.769753, 56.769753,
           56.769756],
          [57.105892, 57.062565, 57.023537, ..., 56.769753, 56.769753,
           56.769756],
          [57.1214  , 57.072567, 57.029255, ..., 56.76975 , 56.769753,
           56.769756]], dtype=float32),
   'all_alphas': array([[0.99380314, 0.994796  , 0.9956425 , ..., 0.999

# **Conformal Predictions with PGBM**


In [31]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [32]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())


Training on CPU


Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31.7545
Estimator 

Estimator 29/300, Train metric: 28.3471
Estimator 30/300, Train metric: 27.8655
Estimator 31/300, Train metric: 27.3146
Estimator 32/300, Train metric: 26.7418
Estimator 33/300, Train metric: 26.1088
Estimator 34/300, Train metric: 25.2632
Estimator 35/300, Train metric: 24.6503
Estimator 36/300, Train metric: 24.2793
Estimator 37/300, Train metric: 23.8609
Estimator 38/300, Train metric: 23.3188
Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688


Estimator 69/300, Train metric: 15.2547
Estimator 70/300, Train metric: 15.1427
Estimator 71/300, Train metric: 15.0232
Estimator 72/300, Train metric: 14.7207
Estimator 73/300, Train metric: 14.5002
Estimator 74/300, Train metric: 14.4790
Estimator 75/300, Train metric: 14.3939
Estimator 76/300, Train metric: 14.2124
Estimator 77/300, Train metric: 14.1670
Estimator 78/300, Train metric: 14.0933
Estimator 79/300, Train metric: 13.9494
Estimator 80/300, Train metric: 13.9759
Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332
Estimator 85/300, Train metric: 13.2440
Estimator 86/300, Train metric: 13.0907
Estimator 87/300, Train metric: 13.0373
Estimator 88/300, Train metric: 13.0103
Estimator 89/300, Train metric: 12.8905
Estimator 90/300, Train metric: 12.8355
Estimator 91/300, Train metric: 12.7364
Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648


Estimator 106/300, Train metric: 11.1323
Estimator 107/300, Train metric: 11.1133
Estimator 108/300, Train metric: 11.0399
Estimator 109/300, Train metric: 10.8530
Estimator 110/300, Train metric: 10.7520
Estimator 111/300, Train metric: 10.6698
Estimator 112/300, Train metric: 10.6707
Estimator 113/300, Train metric: 10.5762
Estimator 114/300, Train metric: 10.5468
Estimator 115/300, Train metric: 10.4995
Estimator 116/300, Train metric: 10.4526
Estimator 117/300, Train metric: 10.4320
Estimator 118/300, Train metric: 10.4033
Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300, Train metric: 9.9513
Estimator 125/300, Train metric: 9.9225
Estimator 126/300, Train metric: 9.8781
Estimator 127/300, Train metric: 9.8022
Estimator 128/300, Train metric: 9.8036
Estimator 129/300, Train metric: 9.6728
Estimator 130/300, Trai

Estimator 134/300, Train metric: 9.4700
Estimator 135/300, Train metric: 9.4091
Estimator 136/300, Train metric: 9.4051
Estimator 137/300, Train metric: 9.2752
Estimator 138/300, Train metric: 9.1582
Estimator 139/300, Train metric: 9.1144
Estimator 140/300, Train metric: 9.1211
Estimator 141/300, Train metric: 9.0928
Estimator 142/300, Train metric: 9.0618
Estimator 143/300, Train metric: 9.0502
Estimator 144/300, Train metric: 8.9818
Estimator 145/300, Train metric: 8.9173
Estimator 146/300, Train metric: 8.8772
Estimator 147/300, Train metric: 8.7642
Estimator 148/300, Train metric: 8.7541
Estimator 149/300, Train metric: 8.7641
Estimator 150/300, Train metric: 8.7435
Estimator 151/300, Train metric: 8.6394
Estimator 152/300, Train metric: 8.6179
Estimator 153/300, Train metric: 8.5644
Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496


Estimator 165/300, Train metric: 8.1188
Estimator 166/300, Train metric: 8.1135
Estimator 167/300, Train metric: 8.0945
Estimator 168/300, Train metric: 7.9866
Estimator 169/300, Train metric: 7.9883
Estimator 170/300, Train metric: 7.9698
Estimator 171/300, Train metric: 7.9719
Estimator 172/300, Train metric: 7.9138
Estimator 173/300, Train metric: 7.8723
Estimator 174/300, Train metric: 7.8243
Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684
Estimator 185/300, Train metric: 7.4722
Estimator 186/300, Train metric: 7.4772
Estimator 187/300, Train metric: 7.3982
Estimator 188/300, Train metric: 7.3940
Estimator 189/300, Train metric: 7.3914


Estimator 204/300, Train metric: 7.0687
Estimator 205/300, Train metric: 7.0643
Estimator 206/300, Train metric: 7.0131
Estimator 207/300, Train metric: 7.0047
Estimator 208/300, Train metric: 6.9527
Estimator 209/300, Train metric: 6.9280
Estimator 210/300, Train metric: 6.9172
Estimator 211/300, Train metric: 6.9223
Estimator 212/300, Train metric: 6.8699
Estimator 213/300, Train metric: 6.8710
Estimator 214/300, Train metric: 6.8053
Estimator 215/300, Train metric: 6.7838
Estimator 216/300, Train metric: 6.7737
Estimator 217/300, Train metric: 6.7405
Estimator 218/300, Train metric: 6.6536
Estimator 219/300, Train metric: 6.6429
Estimator 220/300, Train metric: 6.6378
Estimator 221/300, Train metric: 6.6014
Estimator 222/300, Train metric: 6.6011
Estimator 223/300, Train metric: 6.6056
Estimator 224/300, Train metric: 6.5337
Estimator 225/300, Train metric: 6.4972
Estimator 226/300, Train metric: 6.4960
Estimator 227/300, Train metric: 6.4901
Estimator 228/300, Train metric: 6.4128


Estimator 242/300, Train metric: 6.2581
Estimator 243/300, Train metric: 6.2621
Estimator 244/300, Train metric: 6.2624
Estimator 245/300, Train metric: 6.2135
Estimator 246/300, Train metric: 6.2100
Estimator 247/300, Train metric: 6.2147
Estimator 248/300, Train metric: 6.1478
Estimator 249/300, Train metric: 6.1009
Estimator 250/300, Train metric: 6.0913
Estimator 251/300, Train metric: 6.0789
Estimator 252/300, Train metric: 6.0865
Estimator 253/300, Train metric: 6.0385
Estimator 254/300, Train metric: 6.0455
Estimator 255/300, Train metric: 6.0139
Estimator 256/300, Train metric: 5.9809
Estimator 257/300, Train metric: 5.9833
Estimator 258/300, Train metric: 5.9267
Estimator 259/300, Train metric: 5.8744
Estimator 260/300, Train metric: 5.8320
Estimator 261/300, Train metric: 5.8273
Estimator 262/300, Train metric: 5.8217
Estimator 263/300, Train metric: 5.8125
Estimator 264/300, Train metric: 5.8159
Estimator 265/300, Train metric: 5.8148
Estimator 266/300, Train metric: 5.7728


Estimator 280/300, Train metric: 5.5418
Estimator 281/300, Train metric: 5.5485
Estimator 282/300, Train metric: 5.5233
Estimator 283/300, Train metric: 5.5002
Estimator 284/300, Train metric: 5.5035
Estimator 285/300, Train metric: 5.4777
Estimator 286/300, Train metric: 5.4418
Estimator 287/300, Train metric: 5.3929
Estimator 288/300, Train metric: 5.3943
Estimator 289/300, Train metric: 5.3629
Estimator 290/300, Train metric: 5.3620
Estimator 291/300, Train metric: 5.3140
Estimator 292/300, Train metric: 5.3226
Estimator 293/300, Train metric: 5.3074
Estimator 294/300, Train metric: 5.2770
Estimator 295/300, Train metric: 5.2490
Estimator 296/300, Train metric: 5.2060
Estimator 297/300, Train metric: 5.1671
Estimator 298/300, Train metric: 5.1673
Estimator 299/300, Train metric: 5.1422


         0.05         0.1        0.15         0.2         0.3         0.4  \
0   89.368225   89.712395   89.884720   90.171989   90.389870   90.629974   
1  144.479233  144.579300  144.624924  144.643768  144.674820  144.697357   
2   98.188240   98.245857   98.256577   98.268898   98.295525   98.308388   
3  265.277527  265.480011  266.200867  266.369965  266.852905  267.042664   
4  164.376160  164.866272  165.195801  165.458145  165.614960  165.836090   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0   90.748276   90.925255   91.048500   91.379402   91.630531   91.847900   
1  144.707794  144.732162  144.761765  144.819336  144.833145  144.863419   
2   98.317719   98.333893   98.358498   98.378815   98.383484   98.409103   
3  267.218536  267.382019  267.530884  267.831787  267.954651  268.276886   
4  165.930878  166.118423  166.223175  166.377274  166.622299  166.884384   

         0.95  Actual  
0   92.517372   73.07  
1  144.915009  141.67  
2 

In [33]:
# ACP execution for PGBM
best_params = get_best_model_params(best_scores_autosampler, "PGBM")
run_full_acp_pipeline(
    model_class=PGBMWrapper,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/PGBM.xlsx",
)


Training on CPU
Estimator 0/300, Train metric: 54.7747
Estimator 1/300, Train metric: 53.7145
Estimator 2/300, Train metric: 53.0059
Estimator 3/300, Train metric: 51.3243
Estimator 4/300, Train metric: 50.7406
Estimator 5/300, Train metric: 49.6549
Estimator 6/300, Train metric: 49.4859
Estimator 7/300, Train metric: 47.8585
Estimator 8/300, Train metric: 46.2836
Estimator 9/300, Train metric: 45.0995
Estimator 10/300, Train metric: 44.4666
Estimator 11/300, Train metric: 43.1999
Estimator 12/300, Train metric: 42.0482
Estimator 13/300, Train metric: 41.2808
Estimator 14/300, Train metric: 40.8328
Estimator 15/300, Train metric: 39.9602
Estimator 16/300, Train metric: 39.1099
Estimator 17/300, Train metric: 38.0729
Estimator 18/300, Train metric: 37.4134
Estimator 19/300, Train metric: 36.4206
Estimator 20/300, Train metric: 35.9373
Estimator 21/300, Train metric: 35.2499
Estimator 22/300, Train metric: 34.3041
Estimator 23/300, Train metric: 34.0379
Estimator 24/300, Train metric: 33

Estimator 46/300, Train metric: 26.4676
Estimator 47/300, Train metric: 25.9868
Estimator 48/300, Train metric: 25.5099
Estimator 49/300, Train metric: 25.2747
Estimator 50/300, Train metric: 24.6957
Estimator 51/300, Train metric: 24.3077
Estimator 52/300, Train metric: 23.8199
Estimator 53/300, Train metric: 23.6771
Estimator 54/300, Train metric: 23.5669


Estimator 55/300, Train metric: 23.3082
Estimator 56/300, Train metric: 23.2770
Estimator 57/300, Train metric: 22.8948
Estimator 58/300, Train metric: 22.7230
Estimator 59/300, Train metric: 22.2719
Estimator 60/300, Train metric: 22.2237
Estimator 61/300, Train metric: 22.1775
Estimator 62/300, Train metric: 22.1833
Estimator 63/300, Train metric: 22.1063
Estimator 64/300, Train metric: 21.7386
Estimator 65/300, Train metric: 21.6751
Estimator 66/300, Train metric: 21.5513
Estimator 67/300, Train metric: 21.4981
Estimator 68/300, Train metric: 21.5105
Estimator 69/300, Train metric: 21.5363
Estimator 70/300, Train metric: 21.5018
Estimator 71/300, Train metric: 21.1557
Estimator 72/300, Train metric: 20.7645
Estimator 73/300, Train metric: 20.5034
Estimator 74/300, Train metric: 20.4602
Estimator 75/300, Train metric: 20.3694
Estimator 76/300, Train metric: 20.3785
Estimator 77/300, Train metric: 20.1172
Estimator 78/300, Train metric: 19.9580
Estimator 79/300, Train metric: 19.7360


Estimator 108/300, Train metric: 16.7343
Estimator 109/300, Train metric: 16.7351
Estimator 110/300, Train metric: 16.5802
Estimator 111/300, Train metric: 16.4329
Estimator 112/300, Train metric: 16.4480
Estimator 113/300, Train metric: 16.2842
Estimator 114/300, Train metric: 16.1161
Estimator 115/300, Train metric: 15.9833


Estimator 116/300, Train metric: 15.7634
Estimator 117/300, Train metric: 15.7068
Estimator 118/300, Train metric: 15.7179
Estimator 119/300, Train metric: 15.5842
Estimator 120/300, Train metric: 15.6062
Estimator 121/300, Train metric: 15.6139
Estimator 122/300, Train metric: 15.4914
Estimator 123/300, Train metric: 15.5321
Estimator 124/300, Train metric: 15.5284
Estimator 125/300, Train metric: 15.3563
Estimator 126/300, Train metric: 15.2512
Estimator 127/300, Train metric: 15.2302
Estimator 128/300, Train metric: 15.1751
Estimator 129/300, Train metric: 15.0306
Estimator 130/300, Train metric: 14.8929
Estimator 131/300, Train metric: 14.7069
Estimator 132/300, Train metric: 14.6139
Estimator 133/300, Train metric: 14.5794
Estimator 134/300, Train metric: 14.5824
Estimator 135/300, Train metric: 14.4329
Estimator 136/300, Train metric: 14.3117
Estimator 137/300, Train metric: 14.3236
Estimator 138/300, Train metric: 14.1933
Estimator 139/300, Train metric: 14.1078
Estimator 140/30

Estimator 164/300, Train metric: 12.5587
Estimator 165/300, Train metric: 12.4492
Estimator 166/300, Train metric: 12.3571
Estimator 167/300, Train metric: 12.2256
Estimator 168/300, Train metric: 12.2351
Estimator 169/300, Train metric: 12.1696
Estimator 170/300, Train metric: 12.0821
Estimator 171/300, Train metric: 12.0069


... (4 outputs left out)

Estimator 278/300, Train metric: 7.8692
Estimator 279/300, Train metric: 7.8286
Estimator 280/300, Train metric: 7.8301
Estimator 281/300, Train metric: 7.8198
Estimator 282/300, Train metric: 7.7606
Estimator 283/300, Train metric: 7.7378
Estimator 284/300, Train metric: 7.6892
Estimator 285/300, Train metric: 7.6975
Estimator 286/300, Train metric: 7.6617
Estimator 287/300, Train metric: 7.6475
Estimator 288/300, Train metric: 7.5615
Estimator 289/300, Train metric: 7.5346
Estimator 290/300, Train metric: 7.5177
Estimator 291/300, Train metric: 7.5176
Estimator 292/300, Train metric: 7.4914
Estimator 293/300, Train metric: 7.4484
Estimator 294/300, Train metric: 7.4426
Estimator 295/300, Train metric: 7.4532
Estimator 296/300, Train metric: 7.4050
Estimator 297/300, Train metric: 7.2729
Estimator 298/300, Train metric: 7.2818
Estimator 299/300, Train metric: 7.2884


PGBM best lambda: 50
PGBM best run: 0
PGBM ACP alpha_hat: 0.9644
PGBM ACP width: 43.3151
PGBM ACP coverage: 59.57%
PGBM ACP RMSE: 25.7998
PGBM ACP MAE: 20.8342
PGBM ACP target coverage (1 - alpha~): 3.56%  |  miss / alpha~: 0.419  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   1         51.718018       0.999000         0.001000       0.574468 41.727863         0.425958
   20.0                   1         61.708015       0.999000         0.001000       0.574468 41.727863         0.425958
   50.0                   1         91.497551       0.964429         0.035571       0.595745 43.315083         0.419166


{'model': PGBMWrapper(),
 'all_results': {10.0: {'all_losses': array([[51.897964, 51.863342, 51.833904, ..., 51.71802 , 51.718018,
           51.718018],
          [51.90975 , 51.88499 , 51.862667, ..., 51.718014, 51.718014,
           51.718018],
          [51.91863 , 51.890736, 51.865993, ..., 51.718014, 51.718014,
           51.718018]], dtype=float32),
   'all_sizes': array([[41.95993 , 41.915375, 41.877476, ..., 41.728016, 41.728012,
           41.728016],
          [41.975086, 41.943245, 41.914516, ..., 41.72802 , 41.728016,
           41.72802 ],
          [41.986515, 41.950638, 41.9188  , ..., 41.728016, 41.728016,
           41.72802 ]], dtype=float32),
   'all_alphas': array([[0.99380314, 0.9947961 , 0.9956426 , ..., 0.999     , 0.999     ,
           0.999     ],
          [0.9934659 , 0.99417496, 0.9948156 , ..., 0.999     , 0.999     ,
           0.999     ],
          [0.9932115 , 0.9940095 , 0.9947188 , ..., 0.999     , 0.999     ,
           0.999     ]], dtype=float32)

# **Conformal Predictions with TabNet**


In [34]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')
print(best_params)


{'n_d': 32, 'n_a': 64, 'n_steps': 10, 'gamma': 1.3, 'lambda_sparse': 0.0001, 'optimizer_params': {'lr': 0.02}, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 1, 'scheduler_params': {'step_size': 10, 'gamma': 0.9}, 'scheduler_fn': <class 'torch.optim.lr_scheduler.StepLR'>, 'seed': 42, 'verbose': 0}


In [35]:
class TabNetRegressorCP(TabNetRegressor):
    def fit(self, X, y, *args, **kwargs):
        y_arr = np.asarray(y)
        if y_arr.ndim == 1:
            y_arr = y_arr.reshape(-1, 1)
        return super().fit(X, y_arr, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        preds = super().predict(X, *args, **kwargs)
        return np.asarray(preds).reshape(-1)


best_params = get_best_model_params(best_scores_autosampler, "TabNet")
if isinstance(best_params, dict) and "verbose" in best_params:
    best_params = dict(best_params)
    best_params.pop("verbose", None)

run_full_acp_pipeline(
    TabNetRegressorCP,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "TabNet",
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/TabNet.xlsx",
)


epoch 0  | loss: 20360.84766|  0:00:00s


epoch 1  | loss: 19506.45703|  0:00:00s
epoch 2  | loss: 18600.80078|  0:00:00s
epoch 3  | loss: 17698.73242|  0:00:00s
epoch 4  | loss: 17050.56836|  0:00:00s
epoch 5  | loss: 16185.43848|  0:00:00s


epoch 6  | loss: 15762.7793|  0:00:00s


epoch 7  | loss: 15188.63965|  0:00:00s
epoch 8  | loss: 14603.1748|  0:00:00s
epoch 9  | loss: 13964.03027|  0:00:00s
epoch 10 | loss: 13116.67578|  0:00:00s
epoch 11 | loss: 12160.83789|  0:00:00s


epoch 12 | loss: 11247.4502|  0:00:00s


epoch 13 | loss: 10508.07227|  0:00:00s
epoch 14 | loss: 9723.38086|  0:00:00s
epoch 15 | loss: 8812.28125|  0:00:00s
epoch 16 | loss: 8075.72412|  0:00:00s
epoch 17 | loss: 7358.18604|  0:00:00s


... (27 outputs left out)

epoch 96 | loss: 1.70077 |  0:00:03s
epoch 97 | loss: 2.77004 |  0:00:03s
epoch 98 | loss: 4.62327 |  0:00:03s
epoch 99 | loss: 2.35517 |  0:00:03s


TabNet best lambda: 50
TabNet best run: 0
TabNet ACP alpha_hat: 0.8853
TabNet ACP width: 39.0635
TabNet ACP coverage: 55.32%
TabNet ACP RMSE: 38.9234
TabNet ACP MAE: 26.8704
TabNet ACP target coverage (1 - alpha~): 11.47%  |  miss / alpha~: 0.505  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         44.344589       0.999000         0.001000       0.425532 34.354465         0.575043
   20.0                   1         54.334591       0.999000         0.001000       0.425532 34.354465         0.575043
   50.0                   1         83.269814       0.885269         0.114731       0.553191 39.063511         0.504715


{'model': TabNetRegressorCP(n_d=32, n_a=64, n_steps=10, gamma=1.3, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=1, n_shared=1, epsilon=1e-15, momentum=0.02, lambda_sparse=0.0001, seed=42, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.02}, scheduler_fn=<class 'torch.optim.lr_scheduler.StepLR'>, scheduler_params={'step_size': 10, 'gamma': 0.9}, mask_type='entmax', input_dim=14, output_dim=1, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[], compile_backend=''),
 'all_results': {10.0: {'all_losses': array([[44.483536, 44.456757, 44.433994, ..., 44.344597, 44.344597,
           44.344597],
          [44.492657, 44.473507, 44.45624 , ..., 44.34459 , 44.344593,
           44.34459 ],
          [44.499474, 44.47787 , 44.458706, ..., 44.344593, 44.344593,
           44.34459 ]], dtype=float32),
   'all_sizes': array([[34.5455  , 34.508797, 34.477573, ..., 34.354588, 34.354588,
           34.35459 ],
   